# LABORATORIO DIRIGIDO N.° 03
**Semana 3 · SQL Avanzado — Joins y Agrupaciones**  
Clientes, pedidos y productos de TecnoAndina Retail con JOIN, agrupaciones y vistas en SQLite  
Curso: Fundamentos de Gestión de Datos · Periodo: 2026-II · Docente: Pilar Rocío Sayán Mejía  
Datos: `LD3_TecnoAndina_Retail.xlsx` (GitHub: Rociosayan/Repo_FDG_2026-2, carpeta semana03/datos/)

## Caso de estudio: TecnoAndina Retail

TecnoAndina Retail es una cadena peruana de tecnología con tiendas en Lima, Arequipa, Trujillo y Piura y un canal de venta por Web. Su información está repartida en cuatro tablas y el área comercial necesita saber qué clientes compran más, cuáles todavía no compran y qué productos y categorías sustentan las ventas.

Naturaleza de los datos. Conjunto de práctica inventado (no son ventas ni personas reales), con pedidos del 1 de agosto al 30 de septiembre de 2026, publicado en GitHub en LD3_TecnoAndina_Retail.xlsx. Sirve para practicar consultas, no para afirmar tendencias ni rentabilidad.

- clientes: id_cliente, nombre, ciudad. Una fila es un cliente registrado.

- productos: id_producto, nombre_producto, categoria, precio_lista. Una fila es un producto del catálogo.

- pedidos: id_pedido, id_cliente, fecha_pedido, canal, total. Una fila es un pedido de un cliente.

- detalles_pedido: id_detalle, id_pedido, id_producto, cantidad, precio_unitario. Una fila es un producto incluido en un pedido.

Las tablas se relacionan por id_cliente (clientes–pedidos), id_pedido (pedidos–detalles_pedido) e id_producto (productos–detalles_pedido).

### Preparación del entorno

Abra el notebook en Google Colab y ejecute la celda 1: importa sqlite3 y pandas, crea la base en memoria, carga las cuatro hojas del Excel desde GitHub y cuenta los registros. Cada resultado se muestra con pd.read_sql_query; pandas muestra los NULL como NaN o None. Si reinicia el entorno, vuelva a ejecutar la celda 1.

**Celda 1 de 12**

In [ ]:
import sqlite3
import pandas as pd

# Base de práctica de TecnoAndina Retail (agosto y septiembre de 2026)
url = "https://raw.githubusercontent.com/Rociosayan/Repo_FDG_2026-2/main/semana03/datos/LD3_TecnoAndina_Retail.xlsx"
conexion = sqlite3.connect(":memory:")

# Cada hoja del Excel pasa a una tabla de SQLite
pd.read_excel(url, sheet_name="clientes").to_sql("clientes", conexion, index=False)
pd.read_excel(url, sheet_name="productos").to_sql("productos", conexion, index=False)
pd.read_excel(url, sheet_name="pedidos").to_sql("pedidos", conexion, index=False)
pd.read_excel(url, sheet_name="detalles_pedido").to_sql("detalles_pedido", conexion, index=False)

consulta = """SELECT
  (SELECT COUNT(*) FROM clientes) AS clientes,
  (SELECT COUNT(*) FROM productos) AS productos,
  (SELECT COUNT(*) FROM pedidos) AS pedidos,
  (SELECT COUNT(*) FROM detalles_pedido) AS detalles_pedido;"""
pd.read_sql_query(consulta, conexion)

,clientes,productos,pedidos,detalles_pedido
0,24,14,42,68


## A. Relacionar tablas

### Ejercicio 1 — INNER JOIN y LEFT JOIN: clientes con y sin pedidos

El área comercial quiere saber cuántos clientes compraron y quiénes todavía no registran pedidos. INNER JOIN conserva solo las filas con pareja; LEFT JOIN conserva todas las filas de la tabla izquierda y completa con NULL.

Se solicita. Ejecute las celdas 2 y 3.

**Celda 2 de 12**

In [ ]:
consulta = """SELECT COUNT(p.id_pedido) AS pedidos_unidos,
       COUNT(DISTINCT c.id_cliente) AS clientes_con_pedidos,
       (SELECT COUNT(*) FROM clientes) AS clientes_registrados
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

,pedidos_unidos,clientes_con_pedidos,clientes_registrados
0,42,18,24


**Celda 3 de 12**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
ORDER BY c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad
0,6,Andrés Villanueva,Lima
1,11,Raúl Benavides,Arequipa
2,15,Héctor Burga,Trujillo
3,20,Marta Infante,Piura
4,22,Gabriela Ponce,Arequipa
5,24,Natalia Ato,Piura


**Pregunta 1.** ¿Cuántos pedidos y cuántos clientes distintos devuelve la celda 2? ¿Por qué INNER JOIN no muestra a todos los clientes registrados?

**Respuesta:** _Devuelve 42 pedidos (pedidos_unidos) y 18 clientes distintos (clientes_con_pedidos). Porque el INNER JOIN solo conserva las filas que tienen coincidencia en ambas tablas. Esto significa que solo incluye a los clientes que han realizado al menos un pedido. Como hay 6 clientes que no tienen ningún pedido registrado, son excluidos del resultado._

**Pregunta 2.** En la celda 3, ¿qué tabla es la izquierda y por qué se filtra con p.id_pedido IS NULL y no con = NULL? ¿Cuántos clientes no tienen pedidos?

**Respuesta:** _La tabla de la izquierda es clientes, ya que es la que se define primero en la cláusula FROM antes del operador LEFT JOIN. En SQL, NULL representa la ausencia de un valor o un valor desconocido, por lo que no se puede comparar con operadores tradicionales como el =. Cualquier comparación usando = NULL dará como resultado un valor nulo/falso. Para comprobar correctamente si un campo está vacío o no tiene correspondencia, se debe utilizar obligatoriamente el operador especial IS NULL. No tienen pedidos 6 clientes (tal como se muestra en la tabla de resultados de la celda 3, con los índices del 0 al 5).._

### Ejercicio 2 — RIGHT JOIN y FULL OUTER JOIN

Compras pregunta qué productos no se vendieron y la jefatura quiere comparar a los clientes entre agosto y septiembre. RIGHT JOIN conserva la tabla derecha; FULL OUTER JOIN conserva las filas de ambas tablas. Ambos requieren SQLite 3.39.0 o posterior.

Se solicita. Ejecute la celda 4 (versión y RIGHT JOIN) y la celda 5, que crea un resumen por cliente para cada mes y los une con FULL OUTER JOIN.

**Celda 4 de 12**

In [ ]:
# RIGHT JOIN y FULL OUTER JOIN requieren SQLite 3.39.0 o posterior
consulta = """SELECT sqlite_version() AS version_sqlite;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT pr.id_producto, pr.nombre_producto, pr.categoria
FROM detalles_pedido AS d
RIGHT JOIN productos AS pr ON d.id_producto = pr.id_producto
WHERE d.id_detalle IS NULL
ORDER BY pr.id_producto;"""
display(pd.read_sql_query(consulta, conexion))

,version_sqlite
0,3.45.1


,id_producto,nombre_producto,categoria
0,5,Impresora multifuncional,Cómputo
1,14,Reloj inteligente,Accesorios


**Celda 5 de 12**

In [ ]:
conexion.executescript("""
DROP VIEW IF EXISTS resumen_agosto;
DROP VIEW IF EXISTS resumen_septiembre;
CREATE TEMP VIEW resumen_agosto AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-08-01' AND '2026-08-31'
GROUP BY id_cliente;
CREATE TEMP VIEW resumen_septiembre AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-09-01' AND '2026-09-30'
GROUP BY id_cliente;
""")
consulta = """SELECT COALESCE(a.id_cliente, s.id_cliente) AS id_cliente,
       a.monto AS monto_agosto, s.monto AS monto_septiembre
FROM resumen_agosto AS a
FULL OUTER JOIN resumen_septiembre AS s ON a.id_cliente = s.id_cliente
ORDER BY id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,monto_agosto,monto_septiembre
0,1,1435.0,1967.0
1,2,1522.0,2615.0
2,3,656.0,840.0
3,4,2452.0,NaN
4,5,NaN,640.0
5,7,165.0,3460.0
6,8,271.0,236.0
7,9,220.0,NaN
8,10,6400.0,723.0
9,12,NaN,1420.0


**Pregunta 3.** ¿Qué versión de SQLite informó la celda 4? En el RIGHT JOIN, ¿qué tabla es la derecha y qué productos no se vendieron?

**Respuesta:** _Informó la versión 3.45.1.La tabla de la derecha es productos (ya que se define inmediatamente después de la palabra clave RIGHT JOIN). Los productos que no se vendieron son:

	• Impresora multifuncional (id_producto: 5)
	• Reloj inteligente (id_producto: 14)

**Pregunta 4.** En la celda 5, ¿cuántos clientes compraron solo en agosto, solo en septiembre y en ambos meses? ¿Cómo los reconoce por los NULL?

**Respuesta:** _
• Solo en agosto: 3 clientes (IDs: 4, 9, 13).
• Solo en septiembre: 3 clientes (IDs: 5, 12, 18).
• En ambos meses: 12 clientes (IDs: 1, 2, 3, 7, 8, 10, 14, 16, 17, 19, 21, 23).

• Se reconoce que compraron solo en agosto porque la columna monto_septiembre tiene un valor nulo o vacío (NaN/NULL).
• Se reconoce que compraron solo en septiembre porque la columna monto_agosto tiene un valor nulo o vacío (NaN/NULL).
• Se reconoce que compraron en ambos meses porque ninguna de las dos columnas de montos es NULL (ambas tienen valores numéricos válidos)._

## B. Agrupar y filtrar grupos

### Ejercicio 3 — WHERE, GROUP BY y HAVING

La jefatura pide los clientes con al menos dos pedidos en septiembre. WHERE filtra filas antes de agrupar; GROUP BY forma un grupo por cliente; HAVING filtra los grupos ya formados.

Se solicita. Ejecute la celda 6.

**Celda 6 de 12**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre,
       COUNT(p.id_pedido) AS pedidos_septiembre,
       SUM(p.total) AS monto_septiembre
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.fecha_pedido >= '2026-09-01'
GROUP BY c.id_cliente, c.nombre
HAVING COUNT(p.id_pedido) >= 2
ORDER BY monto_septiembre DESC, c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,pedidos_septiembre,monto_septiembre
0,2,Carlos Mendoza,3,2615
1,1,Lucía Paredes,2,1967
2,12,Sofía Linares,2,1420
3,17,Luis Távara,2,885
4,10,Patricia Zevallos,2,723


**Pregunta 5.** ¿Qué filtra WHERE y qué filtra HAVING? Un cliente con tres pedidos en total, pero uno solo en septiembre, ¿aparece? Justifique.

**Respuesta:** _• WHERE filtra las filas individuales antes de que se agrupen. En este caso, descarta todos los registros que no pertenezcan al mes de septiembre (p.fecha_pedido >= '2026-09-01').
• HAVING filtra los grupos ya formados por la cláusula GROUP BY. En este caso, evalúa el resultado de la función de agregación para conservar solo a los clientes cuyo conteo final de pedidos sea mayor o igual a dos (COUNT(p.id_pedido) >= 2).

No aparece. Al pasar por el filtro del WHERE, los dos pedidos que no corresponden a septiembre son eliminados de inmediato. Por lo tanto, al momento de agrupar los datos (GROUP BY), ese cliente llega únicamente con 1 pedido. Como el grupo resultante tiene un conteo de 1, es rechazado por la condición del HAVING (COUNT(...) >= 2).

## C. Consultar y reutilizar resultados

### Ejercicio 4 — Subconsulta: pedidos sobre el promedio por pedido

Logística quiere revisar los pedidos que superan el monto de un pedido promedio. La subconsulta calcula primero el promedio; la consulta exterior compara cada pedido con ese valor.

Se solicita. Ejecute la celda 7.

**Celda 7 de 12**

In [ ]:
consulta = """SELECT p.id_pedido, p.fecha_pedido, c.id_cliente, c.nombre, p.total
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente
WHERE p.total > (SELECT AVG(total) FROM pedidos)
ORDER BY p.total DESC, p.id_pedido;"""
pd.read_sql_query(consulta, conexion)

,id_pedido,fecha_pedido,id_cliente,nombre,total
0,3,2026-08-02,10,Patricia Zevallos,6400
1,13,2026-08-17,13,Diego Alva,5655
2,26,2026-09-08,7,Carlos Mendoza,3460
3,31,2026-09-15,18,Ana Girón,3045
4,35,2026-09-24,2,Carlos Mendoza,2544
5,20,2026-08-21,4,Jorge Salazar,2452
6,9,2026-08-12,16,Camila Neyra,2256
7,15,2026-08-18,17,Luis Távara,1680
8,11,2026-08-16,2,Carlos Mendoza,1522
9,28,2026-09-10,1,Lucía Paredes,1485


**Pregunta 7.** ¿Qué devuelve la subconsulta y cuántos pedidos la superan? ¿Qué clientes aparecen más de una vez?

**Respuesta:** _• La subconsulta (SELECT AVG(total) FROM pedidos) devuelve el monto promedio de todos los pedidos registrados.
• Un total de 13 pedidos superan dicho promedio (representados por las 13 filas de la tabla de resultados, con índices del 0 al 12).

• Los clientes que aparecen repetidos en el listado son:
	• Carlos Mendoza (aparece con el id_cliente 2 en los índices 4 y 8; además de un registro independiente con el id_cliente 7 en el índice 2).
	• Lucía Paredes (aparece con el id_cliente 1 en los índices 9 y 11).

### Ejercicio 5 — Vista: guardar el resumen por cliente

Varias áreas piden el mismo resumen por cliente. Una vista guarda la consulta con un nombre y se vuelve a ejecutar cada vez que se consulta.

Se solicita. Ejecute la celda 8, que crea vista_resumen_cliente y la consulta.

**Celda 8 de 12**

In [ ]:
conexion.executescript("""
DROP VIEW IF EXISTS vista_resumen_cliente;
CREATE VIEW vista_resumen_cliente AS
SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       COALESCE(SUM(p.total), 0) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad;
""")
consulta = """SELECT *
FROM vista_resumen_cliente
WHERE nombre = 'Carlos Mendoza' OR ciudad = 'Piura'
ORDER BY ciudad, monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,num_pedidos,monto_acumulado,promedio_pedido
0,7,Carlos Mendoza,Arequipa,2,3625,1812.50
1,2,Carlos Mendoza,Lima,4,4137,1034.25
2,17,Luis Távara,Piura,4,3224,806.00
3,18,Ana Girón,Piura,1,3045,3045.00
4,19,Pedro Seminario,Piura,2,1502,751.00
5,20,Marta Infante,Piura,0,0,NaN
6,24,Natalia Ato,Piura,0,0,NaN


**Pregunta 8.** ¿Qué guarda la vista: los datos o la consulta? ¿Qué columna distingue a los dos clientes llamados Carlos Mendoza?

**Respuesta:** _• La vista guarda la consulta (la estructura y definición de la consulta SQL), no los datos reales. Cada vez que se consulta la vista, esta se vuelve a ejecutar en tiempo real sobre las tablas base para obtener los datos actualizados. • Se distinguen mediante la columna id_cliente, ya que uno posee el ID 7 (con ciudad Arequipa) y el otro posee el ID 2 (con ciudad Lima)._

## Actividad 3 — Informe analítico comparado

La gerencia evalúa ofrecer beneficios a los clientes que más compraron. Ejecute la celda 9, complete la tabla y responda citando cifras. Liderar por monto en dos meses no convierte a un cliente en el más rentable.

**Celda 9 de 12**

In [ ]:
# Clientes con mayor monto acumulado (top cinco)
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       SUM(p.total) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY monto_acumulado DESC, c.id_cliente
LIMIT 5;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,num_pedidos,monto_acumulado,promedio_pedido
0,10,Patricia Zevallos,Arequipa,3,7123,2374.33
1,13,Diego Alva,Trujillo,2,5879,2939.50
2,2,Carlos Mendoza,Lima,4,4137,1034.25
3,7,Carlos Mendoza,Arequipa,2,3625,1812.50
4,1,Lucía Paredes,Lima,4,3402,850.50


| Posición | id_cliente | Nombre | Ciudad | Pedidos | Monto acumulado (S/) | Promedio por pedido (S/) |
|---|---|---|---|---|---|---|
| 1 | 10 | Patricia Zevallos | Arequipa | 3 | 7123 | 2374.33 |
| 2 | 13 | Diego Alva | Trujillo | 2 | 5879 | 2939.50 |
| 3 | 2 | Carlos Mendoza | Lima | 4 | 4137 | 1034.25 |
| 4 | 7 | Carlos Mendoza | Arequipa | 2 | 3625 | 1812.50 |
| 5 | 1 |	Lucía Paredes | Lima | 4 | 3402 | 850.50 |

**A.** ¿Qué clientes lideran por monto acumulado y quién tiene el mayor promedio por pedido?

**Respuesta:** _• Por monto acumulado, los líderes indiscutibles son Patricia Zevallos con S/ 7,123 y Diego Alva con S/ 5,879.
• El mayor promedio por pedido lo tiene Diego Alva con un promedio de S/ 2,939.50 por cada compra individual, superando al resto de la lista a pesar de tener menos pedidos totales._

**B.** ¿Qué información adicional pediría antes de recomendar beneficios? Redacte una recomendación prudente en tres líneas.

**Respuesta:** _• Información adicional: Solicitaría el margen de ganancia neto por producto vendido y el historial de comportamiento de pago o devoluciones de los clientes.
• Recomendación prudente (3 líneas):
Se sugiere evaluar el beneficio comercial priorizando a Diego Alva por su alto ticket promedio de S/ 2,939.50, condicionado a verificar la rentabilidad real de los productos que adquiere. Asimismo, se debe cruzar esta información con el margen neto de cada venta, dado que liderar en volumen de facturación bruta no necesariamente equivale a generar la mayor utilidad para la empresa.
_

## Reto de aplicación y retroalimentación

Indicaciones. Cada reto presenta la consulta ya escrita: ejecútela, identifique qué hace cada cláusula e interprete el resultado en no más de cuatro líneas. Los retos son la base de la actividad calificada de la semana.

### Reto 1 — Clientes sin pedidos por ciudad

Marketing quiere saber en qué ciudades se concentran los clientes que todavía no tienen pedidos. Un cliente sin pedidos no es necesariamente un cliente perdido.

**Celda 10 de 12**

In [ ]:
consulta = """SELECT c.ciudad, COUNT(c.id_cliente) AS clientes_sin_pedidos
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
GROUP BY c.ciudad
ORDER BY clientes_sin_pedidos DESC, c.ciudad;"""
pd.read_sql_query(consulta, conexion)

,ciudad,clientes_sin_pedidos
0,Arequipa,2
1,Piura,2
2,Lima,1
3,Trujillo,1


**Pregunta de cierre.** ¿Cuántos clientes sin pedidos hay y en qué ciudades se concentran? ¿Qué cláusula los conserva y cuál los aísla?

**Respuesta:** _• Hay un total de 6 clientes sin pedidos. Se concentran principalmente en Arequipa (2 clientes) y Piura (2 clientes), seguidos por Lima (1 cliente) y Trujillo (1 cliente)

	• La cláusula que los conserva es LEFT JOIN, ya que mantiene a todos los registros de la tabla izquierda (clientes) aunque no tengan una correspondencia en la tabla derecha (pedidos).
	• La cláusula que los aísla (o filtra) es WHERE p.id_pedido IS NULL, debido a que descarta a todos los clientes que sí realizaron compras y se queda únicamente con aquellos cuyo enlace a un pedido quedó completamente vacío.

### Reto 2 — Ingresos por categoría con un umbral

Finanzas quiere saber qué categorías generaron al menos S/ 10 000 entre agosto y septiembre. El ingreso se calcula desde las líneas de detalle.

**Celda 11 de 12**

In [ ]:
consulta = """SELECT pr.categoria,
       SUM(d.cantidad) AS unidades,
       SUM(d.cantidad * d.precio_unitario) AS ingreso
FROM productos AS pr
INNER JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
GROUP BY pr.categoria
HAVING SUM(d.cantidad * d.precio_unitario) >= 10000
ORDER BY ingreso DESC, pr.categoria;"""
pd.read_sql_query(consulta, conexion)

,categoria,unidades,ingreso
0,Celulares,12,18805
1,Cómputo,37,17662


**Pregunta de cierre.** ¿Qué categorías superan el umbral y cuál queda fuera? ¿Por qué la condición va en HAVING y no en WHERE?

**Respuesta:** _
	• Las categorías que superan el umbral de S/ 10,000 son Celulares (con un ingreso de S/ 18,805) y Cómputo (con un ingreso de S/ 17,662).
	• Queda fuera la categoría Accesorios (y cualquier otra que no aparezca en el resultado de la tabla), debido a que sus ingresos acumulados no alcanzaron el mínimo solicitado.
Porque el filtro del umbral se aplica sobre una función de agregación calculada, en este caso la suma total de ingresos (SUM(d.cantidad * d.precio_unitario)). La cláusula WHERE no puede evaluar funciones agregadas porque actúa fila por fila antes de que los datos se agrupen; obligatoriamente se debe utilizar HAVING, que está diseñado para filtrar los resultados después de realizar el agrupamiento (GROUP BY).

### Reto 3 — Clientes sobre el promedio por cliente

Comercial quiere identificar a los clientes cuyo monto acumulado supera el promedio de los clientes que compraron. La consulta usa la vista de la celda 8 y una subconsulta.

**Celda 12 de 12**

In [ ]:
consulta = """SELECT id_cliente, nombre, ciudad, monto_acumulado
FROM vista_resumen_cliente
WHERE monto_acumulado > (SELECT AVG(monto_acumulado)
                         FROM vista_resumen_cliente
                         WHERE num_pedidos > 0)
ORDER BY monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,monto_acumulado
0,10,Patricia Zevallos,Arequipa,7123
1,13,Diego Alva,Trujillo,5879
2,2,Carlos Mendoza,Lima,4137
3,7,Carlos Mendoza,Arequipa,3625
4,1,Lucía Paredes,Lima,3402
5,17,Luis Távara,Piura,3224
6,18,Ana Girón,Piura,3045
7,16,Camila Neyra,Trujillo,2518
8,4,Jorge Salazar,Lima,2452


**Pregunta de cierre.** ¿Qué valor devuelve la subconsulta y cuántos clientes lo superan? ¿Por qué no debe compararse con el promedio por pedido del ejercicio 4?

**Respuesta:** _

	• La subconsulta devuelve el monto acumulado promedio gastado por cliente (considerando únicamente a aquellos que sí realizaron compras, es decir, WHERE num_pedidos > 0).
	• Un total de 9 clientes superan este promedio general (representados en las 9 filas resultantes de la tabla, con índices del 0 al 8).

• Porque son métricas de análisis completamente distintas que evalúan unidades diferentes:

	• El promedio del ejercicio 4 evalúa el valor de cada compra individual (pedido) en la empresa.
	• El promedio de este Reto 3 evalúa el valor de vida o aporte total del cliente (monto acumulado). Un único cliente puede realizar múltiples pedidos pequeños que individualmente no superan el promedio por pedido, pero que al sumarse todos sí superan con creces el promedio acumulado por cliente.


## Conclusiones

Escriba tres conclusiones técnicas y de negocio de este laboratorio.

1. _Técnica — Precisión en los tipos de JOIN y filtrado de datos: La correcta selección del tipo de JOIN es crucial para la integridad del análisis de negocio; mientras que INNER JOIN aísla únicamente las transacciones efectivas (útil para análisis de ventas y facturación), LEFT JOIN complementado con condiciones IS NULL permite identificar patrones de inactividad o clientes no convertidos sin perder el universo total de la base de datos._

2. _Técnica — Optimización y reutilización con Vistas y Subconsultas: El uso de vistas (CREATE VIEW) permite encapsular la lógica de agregación compleja para su rápida reutilización en distintos reportes corporativos sin duplicar código. Asimismo, combinar vistas con subconsultas para calcular promedios dinámicos permite segmentar automáticamente clientes de alto valor (RFM/LTV) sin recurrir a umbrales fijos o codificados manualmente (hardcoded)._

3. _Negocio — Distinción entre facturación bruta, ticket promedio y rentabilidad: El análisis comercial demuestra que liderar en monto acumulado no siempre equivale a un mayor valor unitario por transacción. Para la toma de decisiones estratégicas (como programas de fidelización), es indispensable equilibrar el volumen total de compras, el ticket promedio por pedido y el margen neto unitario por categoría o producto para asegurar decisiones rentables para la empresa._